In [2]:
import pandas as pd
import requests
import time
from io import StringIO

def scrape_pga_master_sweep(start_year=2000, end_year=2026):
    all_seasons = []
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    
    for year in range(start_year, end_year + 1):
        print(f"Scraping {year}...", end=" ")
        url = f"https://en.wikipedia.org/wiki/{year}_PGA_Tour"
        
        try:
            response = requests.get(url, headers=headers)
            tables = pd.read_html(StringIO(response.text))
            
            target_table = None
            for df in tables:
             
                cols = [str(c).lower() for c in df.columns.get_level_values(0)]
                
              
                if any("winner" in c for c in cols) and any(x in cols for x in ["tournament", "event"]):
                    target_table = df
                    break
            
            if target_table is not None:
         
                if isinstance(target_table.columns, pd.MultiIndex):
                    target_table.columns = [' '.join(col).strip() for col in target_table.columns.values]
                
             
                cols = target_table.columns
                t_col = [c for c in cols if any(x in str(c).lower() for x in ["tournament", "event"])][0]
                w_col = [c for c in cols if "winner" in str(c).lower()][0]
                
            
                temp_df = target_table[[t_col, w_col]].copy()
                temp_df.columns = ['Tournament', 'Winner']
                temp_df['Season'] = year
                
            
                temp_df = temp_df[temp_df['Winner'].str.len() > 2]
                temp_df = temp_df[~temp_df['Winner'].str.contains("Winner|Tournament|Event|Notes", case=False, na=False)]
                
             
                temp_df['Winner'] = temp_df['Winner'].str.replace(r'\[.*?\]', '', regex=True).str.strip()
                temp_df['Tournament'] = temp_df['Tournament'].str.replace(r'\[.*?\]', '', regex=True).str.strip()
                
                all_seasons.append(temp_df)
                print(f"found {len(temp_df)} events.")
            else:
                print("table not found.")
            
            time.sleep(0.5) 
            
        except Exception as e:
            print(f"Error: {e}")

    if all_seasons:
        return pd.concat(all_seasons, ignore_index=True)
    return pd.DataFrame()


master_df = scrape_pga_master_sweep(2000, 2026)

if not master_df.empty:
    master_df.to_csv("pga_full_history_2000_2026.csv", index=False)
    print("\n--- SAMPLE DATA ---")
    print(master_df.head(10))
    print(f"\nTotal rows saved: {len(master_df)}")
else:
    print("\nDataFrame is still empty. Check your internet connection or the URL structure.")

Scraping 2000... found 49 events.
Scraping 2001... found 49 events.
Scraping 2002... found 49 events.
Scraping 2003... found 48 events.
Scraping 2004... found 48 events.
Scraping 2005... found 48 events.
Scraping 2006... found 48 events.
Scraping 2007... found 47 events.
Scraping 2008... found 48 events.
Scraping 2009... found 46 events.
Scraping 2010... found 46 events.
Scraping 2011... found 45 events.
Scraping 2012... found 45 events.
Scraping 2013... found 40 events.
Scraping 2014... found 45 events.
Scraping 2015... found 47 events.
Scraping 2016... found 47 events.
Scraping 2017... found 47 events.
Scraping 2018... found 48 events.
Scraping 2019... found 46 events.
Scraping 2020... found 50 events.
Scraping 2021... found 52 events.
Scraping 2022... found 48 events.
Scraping 2023... found 55 events.
Scraping 2024... found 47 events.
Scraping 2025... found 46 events.
Scraping 2026... found 18 events.

--- SAMPLE DATA ---
                                        Tournament           

In [9]:
import pandas as pd


df1 = pd.read_csv("2021 PGA Tour Filtered Stats copy.csv", encoding='latin1')
df1['Year'] = 2021

df2 = pd.read_csv("PGA TOUR DATA  copy.csv", encoding='latin1')
df2.rename(columns={'Player Name': 'PLAYER NAME'}, inplace=True)

df3 = pd.read_csv("pga_full_history_2000_2026.csv", encoding='latin1')
df3.rename(columns={'Winner': 'PLAYER NAME', 'Season': 'Year'}, inplace=True)

df4 = pd.read_csv("pga_tour_stats_2020 copy.csv", encoding='latin1')
df4['Year'] = 2020

df5 = pd.read_csv("pgafulldata copy.csv", encoding='latin1')

df6 = pd.read_csv("pgatour_raw copy.csv", encoding='latin1')
df6.rename(columns={'NAME': 'PLAYER NAME'}, inplace=True)


merged = pd.merge(df1, df4, on=['PLAYER NAME', 'Year'], how='outer', suffixes=('_f1', '_f4'))
merged = pd.merge(merged, df2, on=['PLAYER NAME', 'Year'], how='outer', suffixes=('', '_f2'))
merged = pd.merge(merged, df3, on=['PLAYER NAME', 'Year'], how='outer', suffixes=('', '_f3'))
merged = pd.merge(merged, df5, on=['PLAYER NAME'], how='outer', suffixes=('', '_f5'))
merged = pd.merge(merged, df6, on=['PLAYER NAME', 'Year'], how='outer', suffixes=('', '_f6'))



cols_to_keep = [c for c in merged.columns if "Unnamed" not in c]
merged = merged[cols_to_keep]


id_cols = ['PLAYER NAME', 'Year']
rest_of_cols = [c for c in merged.columns if c not in id_cols]
merged = merged[id_cols + rest_of_cols]


merged.to_csv("pga_raw_combined_master.csv", index=False)

print("Success! 'pga_raw_combined_master.csv' has been created.")

Success! 'pga_raw_combined_master.csv' has been created.


In [8]:
import os
print(os.listdir())

['.DS_Store', 'pgafulldata copy.csv', 'PGA Tour data raw.ipynb', '2021 PGA Tour Filtered Stats copy.csv', 'PGA TOUR DATA  copy.csv', 'pgatour_raw copy.csv', 'pga_full_history_2000_2026.csv', '.ipynb_checkpoints', 'pga_tour_stats_2020 copy.csv']
